In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
from SCOUT import eval_on_real_data

In [ ]:
# --- Synthetic data generation ---
# We simulate a logistic model:
#   - n patients with iid context vectors x_i in R^d
#   - Labels y_i ~ Bernoulli(sigma(x_i @ theta_star)), where sigma is the logistic function
#   - theta_star is the unknown true parameter; S is a known bound on ||theta_star||
#
# Here we use unit-norm contexts (common in theory), but eval_on_real_data works
# with any X matrix.

np.random.seed(0)

n = 20000   # number of patients (stream length)
d = 10      # feature dimension
S = 3.0     # L2 norm bound on theta_star (used by SCOUT for confidence sets)

# True parameter: unit direction scaled to norm S
true_theta = np.ones(d) / np.sqrt(d) * S

logistic = lambda z: 1.0 / (1.0 + np.exp(-z))

# Random unit vectors as contexts (one per patient)
X = np.random.randn(n, d)
X /= np.linalg.norm(X, axis=1, keepdims=True)

# Binary labels from logistic model with true_theta
probs = logistic(X @ true_theta)
Y = np.random.binomial(1, probs)

print(f"n={n}, d={d}, class balance={Y.mean():.3f}")

In [ ]:
# --- Run on real data generated above ---
# num_perms=2 for speed; increase to 10–20 for stable estimates

results = eval_on_real_data(
    X,
    Y,
    alpha=0.05,
    num_perms=2,
    delta=0.1,
    S=S,
    aggressiveness=10.0,
    out_dir="test_outputs",
    title="test_scout_results",
    debug=False,
)


In [ ]:

print("Results:")
print(f"theta_star: {np.array2string(results['theta_star'], precision=4)}")
print(f"tau_star: {float(results['tau_star']):.4f}")
print(f"opt_test_rate: {float(results['opt_test_rate']):.4f}")
print(f"final test rate: {float(results['all_cum_tests'][0, -1]):.4f}")
print(f"final error rate: {float(results['all_cum_errors'][0, -1]):.4f}")